# Multi-query attention with a shared KV cache

Multi-query attention (MQA) keeps $H$ query heads but a **single** key head and a single value head shared by all of them. The KV cache shrinks by a factor of $H$, which is what lets a server hold many more concurrent sequences.

Write `mqa(x, Wq, Wk, Wv, Wo, n_heads, past_k=None, past_v=None)` in PyTorch: causal self-attention for $T$ new tokens that may extend an existing cache.

## Shapes (all float64)
- `x`: `(B, T, D)`, the new tokens. $H$ = `n_heads` divides $D$; $d_h = D/H$.
- `Wq`: `(D, D)`, `Wk`: `(D, d_h)`, `Wv`: `(D, d_h)`, `Wo`: `(D, D)`. Projections are $XW$ (no transpose, no bias).
- `past_k`, `past_v`: either both `None` (empty cache) or both `(B, P, d_h)`: keys and values of the $P$ earlier tokens.

## Computation
1. $Q = xW_q$ split into $H$ heads (head $h$ = features $h d_h : (h+1) d_h$). New $k = xW_k$, $v = xW_v$, each `(B, T, d_h)`.
2. The cache becomes `K = cat(past_k, k)` and `V = cat(past_v, v)` along time, length $L = P + T$.
3. New token $i$ sits at position $P + i$ and attends to cached positions $j \le P + i$. Scores are $Q_h K^\top / \sqrt{d_h}$; softmax over keys; output $\sum_j a_{ij} V_j$.
4. Concatenate heads in order to `(B, T, D)` and multiply by $W_o$.

Share the single K/V head across query heads by **broadcasting**, not by copying it $H$ times.

## Output
A tuple `(y, K, V)`: `y` is `(B, T, D)`; `K` and `V` are the updated caches, `(B, P + T, d_h)` each.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def mqa(x, Wq, Wk, Wv, Wo, n_heads, past_k=None, past_v=None):
    """Return (y (B,T,D), K cache (B,P+T,dh), V cache (B,P+T,dh))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[1.0, 0.0], [0.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0], [0.0]], "dtype": "float64"}, {"__nd__": [[0.0], [2.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[[0.0, 0.0], [1.0, 0.5378828427399903]]], "dtype": "float64"}, {"__nd__": [[[1.0], [0.0]]], "dtype": "float64"}, {"__nd__": [[[0.0], [2.0]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[1.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0], [0.0]], "dtype": "float64"}, {"__nd__": [[0.0], [2.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, 2, {"__nd__": [[[1.0], [0.0]]], "dtype": "float64"}, {"__nd__": [[[0.0], [2.0]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[1.1553624034969636, 1.1553624034969636]]], "dtype": "float64"}, {"__nd__": [[[1.0], [0.0], [1.0]]], "dtype": "float64"}, {"__nd__": [[[0.0], [2.0], [2.0]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.35877340800391416, 1.5106773081434572, -1.7863312373111822, 1.686613508665752, -0.047317212156137566, -0.7999785843751532, -0.802956718390362, -1.082816516582591], [-0.22364535840745078, 0.8338841795521574, 0.5840637316429703, 0.6382860242561829, -1.6948083917690129, -1.5709621176914044, 1.553803174289652, 0.96888536545666], [2.18321400147264, 1.2098158348986146, -1.0243913350545566, 1.2852724548122736, 0.6283942642703312, 0.21482492767891934, -0.8198733225718833, 0.0027802218072239555], [-0.14699870178042784, 0.8925351116148221, 0.12449694399871385, 1.5847459143396285, -0.7744857532181485, 0.4711025594104292, 0.49884237745245574, -0.864338539814181], [-1.1812660464710163, 1.013579500325613, 0.5174404713814393, -0.1625051850123894, 0.3943673986045515, -0.6753440287843157, 0.2383482020485698, -0.6051266946615304]], [[0.18878846665371848, 0.9860894628834065, -0.33963024501217354, 0.8497738099419226, -1.0341993407801506, -0.2884907696501737, -1.3647425964001083, -0.10058608537896181], [0.07865408332523142, -0.19726266289721334, -0.26767571670742146, 1.3401946901039739, 1.4237928368424078, -1.2156310049465249, 1.1741144932558947, -1.723069612870992], [-0.7344792508370456, -0.1734455959217569, 0.1110606026713016, -1.0474166760042456, -0.7224687794067585, 0.07665381874343688, -0.6934227233827154, -0.6412007892602988], [0.6662634522239711, 0.9317210528359195, -0.0600550310800827, 0.018197912592243334, -0.019598117164741715, -0.4544882932751738, -0.511393214091543, -1.0377445899208706], [0.6483887219942397, -0.08415967031156173, -0.13743939809669994, 0.20054261321582875, 0.5306748425469374, 1.4210345023787507, -0.27354581958930024, -0.4011722662804173]]], "dtype": "float64"}, {"__nd__": [[0.08022940139097874, -0.3382010633293314, 0.5089720470622091, 0.37049760190382225, 0.27756990514456537, -0.04784598274949176, 0.2958328698440968, 0.42008188665544677], [0.19996239977405383, -0.11199309903461217, -0.6740291982753182, 0.3787849415287785, -0.1413003862604277, -0.3372937527329507, 0.3562393528224171, 0.13526857494212466], [-0.5675345805449967, -0.03961654949202101, -0.10769361287055937, 0.3970447221884894, -0.7580487317432436, -0.15422430001318893, -0.055299647181718564, 0.5675601652120901], [-0.43335594364794333, -1.1761727742024397, -0.5536113326341726, 0.1052279051132248, 0.3238368395978932, 0.693417027730887, 0.16037404921604811, -0.3209241214114667], [0.35375583604483535, -0.004423098187900372, -0.08038317784207753, 0.007021018435614407, 0.5098304203640442, -0.3605765477427094, -0.40957091125526146, -0.026608840084152426], [-0.026627980622058706, -0.32578345173567314, -0.3517826035553399, 0.15106366401697763, 0.5351340537328164, 0.2851967491783635, 0.2740484443865569, 0.4982949506470199], [-0.796389090680492, -0.604130734454869, -0.2821592775716189, -0.4780444576139571, -0.3957590897571435, 0.027949601534629526, -0.513755863139161, -0.1773988073581299], [-0.8172618727797603, -0.3895898279822285, 0.5382191543706301, -0.21715803986236903, 0.12965554686361686, 0.28326168566416393, -0.19636094209468752, 0.2316338922127392]], "dtype": "float64"}, {"__nd__": [[0.08581411417152443, 0.2789620782797992], [-0.2542211062386927, -0.5760685270719542], [-0.4088479571402506, -0.22192663310170363], [0.21433147786136733, -0.08172359477985597], [-0.15952252560877414, -0.07187254893090111], [0.4399027466205701, -0.2301995590118482], [0.6045326675660329, -0.49103713830341483], [-0.23689407228561868, 0.34200106043040324]], "dtype": "float64"}, {"__nd__": [[0.2071307496065669, -0.3422037132316109], [-0.0436135953457224, -0.283571890439236], [0.34040851077694123, 0.2158962588107648], [-0.2852437186412679, 0.15887333773042542], [-0.14685841482987552, -0.028766631099707993], [0.025845938010443294, -0.2256176973167099], [0.17000291313931254, 0.18411295913203873], [0.17037266232807757, 0.44185588283843624]], "dtype": "float64"}, {"__nd__": [[-0.10869645700946572, -0.5931437453800777, -0.19562880214388362, 0.08029475570952165, 0.008794285510532419, 0.23147521669002824, 0.3225671189809418, -0.04703595900302689], [-0.04414805581476067, -0.39496301406646944, 0.41323820819655865, -0.10557463011513438, -0.43332307460757014, -0.4457495527131172, -0.16590631065142047, 0.6221596840734833], [0.3090185080477703, -0.6776168944953703, -0.04944577636232287, 0.10558385329734057, 0.25062292990023805, 0.13746512222929297, -0.25629660477923705, 0.16475456342283382], [-0.2216602240508337, -0.8962593694051053, -0.28354539102612647, -0.27438692618977395, 0.3814516035253483, 0.4224018700735706, -0.41811176985665677, 0.06956382128611666], [0.036054116993841714, -0.5743181992380004, -0.8212546626500139, -0.03710616049651438, 0.054577490876747896, -0.23793283004480772, -1.0844665981645416, -0.01179359145268345], [-0.23144794989114953, -0.3557625337090218, -0.129674803875206, -0.451422677294144, 0.22393709122747257, 0.004241545969603937, -0.2237826547828431, 0.11274904305369804], [-0.012840088905789927, -0.6072824820635473, -0.057658702125561845, -0.18286261631296852, -0.6962862708999203, -0.26081699118146917, 0.17053606584516146, -0.24779824913670964], [0.5630029702905982, 0.6792865740579018, -0.122538145544507, 0.2213737697184704, 0.7970610591366059, -0.1326106089400896, -0.35958283349801384, -0.1689327980588324]], "dtype": "float64"}, 4], "e": {"__tup__": [{"__nd__": [[[-0.38960332409282, 4.548569150445873, 1.7273678664226735, 0.727376063193423, -0.5378047875424751, 0.3526466383021639, 2.499478049750159, -0.5067275321348567], [-0.10503271056941485, 0.2875344815064378, 0.40561062294272415, -0.1053742971262919, 0.1352930860274006, -0.07182754955679227, 0.10065131112248571, 0.3463720813993051], [-0.16758859682507188, 1.7769571202050076, 0.990635376873733, 0.41159321214146655, -0.6931975362828937, -0.11171245261962096, 1.282836694322546, 0.04204752378261343], [-0.05299397791278143, 1.4255877871341518, 0.7497780770356468, 0.36546504012782705, -0.3661410199292959, 0.004068448222529386, 0.9499409370310825, -0.020608035573665848], [-0.0741854834833372, 1.1130851088339404, 0.20335132413759022, 0.06608711099733196, 0.16972442336902224, 0.16301640763955713, 0.3579445665823963, -0.243502352855831]], [[-0.13609239681194163, 1.6121557514596394, 0.5837170740165054, 0.31080012216796493, -0.2901080469845462, 0.13391642455788946, 0.9598839149843224, -0.2410314862370117], [-0.08855510588058192, 1.925981117256935, 0.8458595855148292, 0.227381759594191, -0.06272031155537305, 0.03131206249374275, 0.8517414976985409, -0.014635243180642549], [-0.10445790368215027, 1.2347896220041303, 0.4306952644573793, 0.2085580858712966, -0.08454021179213056, 0.12329110039209662, 0.658834920018277, -0.1469544542669066], [-0.029117033113330016, 1.426856978663586, 0.5524126544606375, 0.3118235299562398, -0.22249665048904269, 0.055296840930118656, 0.7557455445367804, -0.13874437683492652], [-0.09167805030663782, 1.264418527337145, 0.4896785762065985, 0.30130706319984657, -0.407723753851138, 0.08739189768316094, 0.8805643573903594, -0.2406197742132766]]], "dtype": "float64"}, {"__nd__": [[[0.16530864072043697, -0.30005841984133635], [-0.04408022087231196, -0.6727155869079642], [0.07204471582155554, 0.34332191160062686], [0.886355208714154, -1.3056469379138822], [-0.6779830326203642, -1.2118450777706884]], [[-0.6766266567655165, 0.26701588671530874], [0.8096716382023299, -0.9028621936769449], [-0.40716617247422887, 0.11146099835344093], [-0.4113572558899428, -0.3367974437116386], [0.6463408174194257, -0.12467265106022093]]], "dtype": "float64"}, {"__nd__": [[[-1.4154659093705846, -1.1132966285592085], [0.5715771976895468, 1.184943746766444], [-0.5415209021124107, -1.3234063711417359], [-0.4155727781293241, -0.2982215210130641], [-0.20433587256698152, 0.12023583926781539]], [[-0.4666314526414011, -0.483422162579769], [-0.782982351779951, -0.1277151617811503], [0.07296119122039585, -0.24940110780527455], [-0.2008763132234346, -0.9518655169498319], [-0.12207578351238836, -0.7593274048725841]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-1.3742214526554224, -1.3491851888620348, 0.2086518433004105, -0.4167563978840294, 0.24644175655585768, 0.4883741105141886], [-0.7643160931773055, -0.1944523423331713, -0.2693043085863964, -0.6686649004393013, -0.8364556720740618, 0.3155505374569986], [-0.14024643621485902, 0.3322867996571848, 1.0009222444961468, 1.0142174316108183, -1.1120591826841941, -0.8558236870860543]]], "dtype": "float64"}, {"__nd__": [[-0.10250057294386117, 0.04945484581435576, 0.37535418345443344, -0.3848708555768067, -0.17474123023494093, 0.09881427787636965], [0.49154604764709336, 0.5516937153287674, -0.40239060164751694, -0.08481489719734563, -0.32310011494497215, 0.33899165122364217], [0.17876002596515597, -0.1431184632112564, -0.24449872909110018, 0.11798093603999325, 0.3921176240248654, -0.3069776576320876], [-0.1294647244508019, 0.045189507275004245, 0.007204746600763575, 0.13514679866359822, -0.39983066350795826, -0.23797104675803174], [-0.8400420610605155, 0.02229346507552709, 0.4280627691524327, -0.8228781616168288, -0.2527231987922699, 0.359409671145247], [-0.057121094613768325, -0.7580171278270177, 0.4675153326977272, -0.5703878551444137, 0.5483641863008083, -0.233856239517776]], "dtype": "float64"}, {"__nd__": [[-0.28261817516269516, 0.01077784232749323], [-0.00887021855641299, 0.24019020914080152], [0.20100242268353938, -0.05917501756698995], [0.5290131314827499, 0.28538992851937145], [0.10951664742140102, 0.05404605697887843], [-0.0381300851125199, -0.03103227787176323]], "dtype": "float64"}, {"__nd__": [[0.26595959735699976, 0.18970132020326358], [0.01589393520622068, 0.4060451693589262], [-0.10488461117954687, 0.17435845502119718], [0.34380951334079585, -0.44620173401422303], [-0.10564594840735056, 0.6675225866313006], [-0.3828414204658236, 0.46083184270006505]], "dtype": "float64"}, {"__nd__": [[-0.42390514090843995, 0.16221415613492202, -0.42567568664746747, -0.24655554596102902, -0.15012489855227168, -0.21952814532031134], [0.40342762356570755, 0.33339771864864376, 0.3480239865722864, 0.07739532285776672, 0.6546766596153786, 0.4115169481158731], [0.12990114599839364, -0.41951429195333034, 0.2741786456645449, 0.26086339951588056, 0.03773697446843997, 0.28515493900255073], [0.30797460970902785, -0.1193838214357062, 0.2261180444263967, 0.1388261960450812, 0.12525986310875079, -0.13671857348617777], [-0.18519117385809658, 0.047449712596139934, 0.5726871430898526, 0.9303941690744557, -0.14792229915886268, -0.4217388375955885], [-0.13662421088881435, -0.334057237583905, -0.16682396048442877, 0.1587128729277394, -0.3657006127606043, 0.2435866840967328]], "dtype": "float64"}, 3, {"__nd__": [[[0.7915542794734144, 1.5164360247500759], [1.8192748316879657, 0.997828907708521], [2.2671920580788893, -0.49962791416033], [0.276470890505262, 0.17205662638670258]]], "dtype": "float64"}, {"__nd__": [[[0.43919349351729436, 1.024591077762065], [-0.3232677308941149, 1.7133716327948385], [1.2759740433101456, 0.35307602330793086], [0.47171827985954007, -0.9634178520124883]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[-0.030387954542777817, -0.35180500159636324, 0.4195639147410928, 0.7842886142821894, -0.269835750293795, -0.17253760793293108], [0.0058863367523467325, -0.019285487274615563, 0.22084045596783078, 0.414910828278993, 0.03982271442002174, 0.010822739341471307], [0.24951423045292753, 0.2273403923829202, 0.2968375678702119, 0.11978146567546799, 0.4860644143596648, 0.14098160728803563]]], "dtype": "float64"}, {"__nd__": [[[0.7915542794734144, 1.5164360247500759], [1.8192748316879657, 0.997828907708521], [2.2671920580788893, -0.49962791416033], [0.276470890505262, 0.17205662638670258], [0.23018517416983403, -0.471993425803082], [-0.2937666668551277, -0.2848367509360431], [0.6852545075102922, 0.2749740753527881]]], "dtype": "float64"}, {"__nd__": [[[0.43919349351729436, 1.024591077762065], [-0.3232677308941149, 1.7133716327948385], [1.2759740433101456, 0.35307602330793086], [0.47171827985954007, -0.9634178520124883], [-0.7651058392816857, -0.196620331830474], [-0.4404529528495189, -0.3854815695124559], [0.6568270233782036, -1.3064232336253996]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-2.4759797049869987, -0.0809123582275483, 0.3239967842305745, 0.23537058308694503, -1.0897890047921537, -0.4421162864665036, 1.759257542559709, 0.7362004871113208]], [[0.6243032398185351, -0.2989905966535996, 0.7467517137104908, 1.2923755482918649, 0.7663819416213754, -0.3931861881713645, 0.29516001191068375, -0.8641129705485112]], [[-0.2917449236407673, 0.6010780516446905, -0.5195588624307894, -0.21733011358290186, 1.4400312683510301, -0.15393592947116394, 0.7754822078311886, -0.2849522608169767]]], "dtype": "float64"}, {"__nd__": [[-0.44673250700666345, 0.0073332139348865205, -0.13732509695639955, -0.7935427567114393, -0.4720476630923173, -0.16132070192390258, 0.5921016639356519, -0.22246639856556666], [-0.0069327534931482694, 0.25589807153620286, 0.12149172236429436, -0.12841369421883578, 0.6812703792788286, 0.5377484970649369, -0.9625214791892307, 0.3872281035430933], [0.22178878162910434, 0.39476709193365417, 0.04135409568468558, 0.3335079316012827, 0.7708133142907695, 0.6852068598248493, 0.7160879460295186, 0.42400083320176746], [0.37236582186956935, 0.3799124430000021, -0.437769763605528, 0.9079080485926824, 0.10043443045686269, 0.1350633692965963, 0.05373987056281159, 0.194265408753013], [-0.8353651726292605, 0.09749182348537562, -0.49643486335754955, -0.3850592389580358, -0.7318052385869162, -0.6757076683656862, 0.238407708488867, -0.18866797967690155], [-0.03565626014561153, -0.06273010190273275, -0.45575965744777974, -0.2764419538118185, -0.49563905833446287, -0.11915917255244889, -0.8778754948854357, -0.1525961509099409], [0.3650926034478392, 0.482501784753116, 0.45754139695822116, 0.19951243275596042, -0.3969943331654818, 0.3623866438876273, 0.2082395766891624, 0.10736679113907349], [-0.48017492127427186, 0.1567656012913251, 0.5623549742885378, 0.29615512277291184, -0.08971689510406605, 0.2179232302168598, 0.0025019443064491207, 0.07181715431327582]], "dtype": "float64"}, {"__nd__": [[0.33694421705895095, 0.13749391195917965, 0.029817707437021453, 0.15364638940143863], [-0.32507869987465576, -0.39531303437992205, 0.19329127147196762, -0.23510141983669303], [0.22283335275882774, -0.7310895707736252, 0.47650862349064904, -0.03933509524231293], [0.6625390260771958, -0.983765858782377, -0.6681342492437148, 0.5586600805348327], [0.06320382945395017, 0.12191080823169397, 0.050735605627082674, 0.06507382986616247], [0.21539775556613014, -0.3994545478999705, -0.0646248312938859, 0.5188052889934051], [0.07389920998449964, -0.2445681699503282, 0.04029477262851172, 0.41741827909477414], [-0.07696126889152052, -0.06828347045996835, 0.19761961103536285, 0.3405703130360547]], "dtype": "float64"}, {"__nd__": [[0.589474528195043, -0.04868482364802727, 0.14130286872515263, 0.4182286683359016], [-0.7530126268434174, 0.06092161816676975, 0.1877642097655726, -0.028735433830754015], [-0.3342037895861365, 0.2377439612187343, 0.49959345225543217, -0.029535797633353136], [-0.1860589995150475, -0.07930072127170568, 0.6258612888866031, 0.5017606376685397], [0.024595426640929695, -0.42858323772031764, -0.495460409637958, -0.11999520493965365], [-0.11701114286680021, 0.11499782674144461, -0.4409806429636602, -0.04045839244506125], [0.22166267620620783, -0.6622397478707275, 0.9243013985274926, 0.23963981627625308], [0.22017390537406498, -0.5080352316958456, -0.3975750028557468, 0.4169506122276067]], "dtype": "float64"}, {"__nd__": [[0.36536049780966007, -0.9585935040234349, 0.47257881019838355, 0.3027848375244374, 0.2613124238530056, -0.4297509462168792, 0.11625204899593355, -0.22808757982181618], [0.23806561818708755, 0.7486148080189361, 0.27043143227001687, 0.42061044771596623, -0.1091555011151484, -0.39225461255203636, 0.0215999876251566, 0.4303668213142314], [0.6030587873637536, -0.006830733523018325, 0.11844599610513262, -0.14171762092621806, -0.6683260163433906, 0.05207703527651775, 0.15673907469520834, -0.1473712540396279], [-0.05077311481962398, -0.7006152780967079, 0.04109404295577824, 0.5703448845746153, 0.00928008279373027, 0.2564365730115536, 0.1958799464690677, -0.2838026775609816], [0.10247846486037263, -0.30831703629124885, 1.183437510054305, 0.10083453348175281, -0.4161370260047436, -0.5941692435555631, 0.6143405603728236, 0.26391928210524923], [0.17702288062100635, -0.5250354783032285, -0.2061404565426583, -0.4662592561331296, 0.4569714753065071, -0.1909293271997271, 0.7875506931094893, 0.225432637563513], [-0.1102995500252471, 0.5536190573602445, -0.8196284612640533, -0.3143005846454687, -0.1422600949903072, 0.444856348717995, 0.4479742662114283, 0.16032643274862968], [-0.6910281232784868, -0.19312515151142698, -0.422545467965433, 0.5166430092585033, -0.15812787791402957, 0.25322433340333256, 0.7593242628596732, -0.011740451161010978]], "dtype": "float64"}, 2, {"__nd__": [[[0.7427328337621717, -0.33831406313040074, 0.0920082045447397, 0.0874176471196743], [0.05983564572452336, 0.11362012650575741, -0.3481247266864322, 1.9105256958354058], [-0.60342889957522, 0.2909082404547313, -0.252476791803004, -2.21433140332863], [1.014081570718048, 0.27340978305092917, -1.3309396416275998, -0.6865437031087761], [-1.5716378217716929, 0.324499439758768, 1.59897831343026, 1.2550088075344976], [-1.255871572366376, -1.6855427638682352, 0.5631978197948958, 1.7180851907698207]], [[0.08434885076115327, 1.0825546501385463, -1.361415899469512, -0.08002323669387834], [0.4712736382815932, -0.38139552372937824, -0.19773819667417775, 0.7119008201186471], [-0.5968431318039679, -0.9665044207563379, 0.7722137763102623, 0.2683662165280098], [-0.9716157988656038, 0.45727707819602825, -0.5376127356953474, 0.6626300876609809], [0.014020724262976783, -0.2643422426079685, -0.750808791874266, 0.22371067624615593], [-0.024490099505006135, 1.9131394543817049, -0.023186931500279076, -1.3266491793246087]], [[2.089943400389317, -2.017714388591036, 1.6711454174837166, -0.8429117726014337], [1.407823181786228, -0.16600172219306608, 0.5610540430931822, 0.04759712891396072], [-0.09339558421005838, 0.5373561080684924, -0.6264275124884702, 0.49340285858683025], [-1.0658353634238578, -0.7171913017637742, -0.49420982623205867, -0.08579065848748894], [1.2279667198701043, -0.8239346781917004, -0.8485469505140043, -1.1130437267387114], [-0.3960882256008402, 0.5714571320641562, -0.7374594503673815, -0.2848802346472962]]], "dtype": "float64"}, {"__nd__": [[[-0.5806098743632441, 0.09148518728344766, -2.0073355821449748, 0.6005646180927717], [0.9309711472595998, -0.8293756561812254, -0.06355302824577985, -0.9029276378925517], [1.0076122042921374, 0.638022103029592, 2.905365754466288, -0.19979401144998388], [-0.8672835241573296, -0.4707098575811255, -1.2472258281739477, 0.21930308734331058], [0.32106353464181553, 0.9361309534630777, 0.41775047298731166, -0.3991506082722122], [1.308841835405336, 0.5965915157528774, 0.8488511521670594, -1.1523891832160071]], [[-0.8151179060392977, -0.6208843954177016, -0.37051363122588304, -2.163180399362114], [1.2257990067736229, -0.18498699546738706, 0.8749902772222725, 0.802044669624543], [1.5252804711174976, 0.6059076613181522, 0.41080477286055705, -0.7030293439623367], [0.16296340347886626, 1.7967351326579435, -1.717788434249212, -0.6591092005876211], [0.32364641617881873, 1.6668239959895288, -1.0442172865679762, -0.39469683180605447], [0.2117067351592105, 3.165755800696517, -0.80280893210664, -0.09042974238578648]], [[0.8122257890093711, -0.6343643651294132, 0.4578946078635953, 0.487082731292979], [-0.28431569175490745, 0.4291280141007714, 1.8021716158207064, 0.20716255069019446], [-0.343070423147098, -0.6623713896275052, 0.31705955904387295, 1.125269154091136], [0.23556134312775395, 0.4000979885722656, 2.7388393432137237, -0.6714810579558491], [-0.9863119291726956, 0.36592425989631094, -0.12245718104423046, 0.6658942999413479], [1.331332445583354, 0.45100369130048223, -0.10280538971522843, 0.6117676927688177]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[0.4122049243428999, -0.3684131134812033, 0.9971840348149869, -0.02804923030098705, 0.06957118227408671, -0.6544744940509062, -0.8203037930468664, -0.21663496944689567]], [[0.48324293396450896, 0.36589444448211494, 1.0359347300055926, -0.5712940541340299, 0.28983346294282136, -1.2431826248588738, 0.8203126586603645, 0.9991997031543098]], [[0.31156152390088554, 0.18717006375875878, -0.6008653012213636, -0.24851044909866965, -0.647940484774728, 0.4625616670537588, 0.7962149240046521, -0.018263698361250715]]], "dtype": "float64"}, {"__nd__": [[[0.7427328337621717, -0.33831406313040074, 0.0920082045447397, 0.0874176471196743], [0.05983564572452336, 0.11362012650575741, -0.3481247266864322, 1.9105256958354058], [-0.60342889957522, 0.2909082404547313, -0.252476791803004, -2.21433140332863], [1.014081570718048, 0.27340978305092917, -1.3309396416275998, -0.6865437031087761], [-1.5716378217716929, 0.324499439758768, 1.59897831343026, 1.2550088075344976], [-1.255871572366376, -1.6855427638682352, 0.5631978197948958, 1.7180851907698207], [-0.6705855476314677, -1.2136470583541872, 0.09731754462208646, 0.4421302471318796]], [[0.08434885076115327, 1.0825546501385463, -1.361415899469512, -0.08002323669387834], [0.4712736382815932, -0.38139552372937824, -0.19773819667417775, 0.7119008201186471], [-0.5968431318039679, -0.9665044207563379, 0.7722137763102623, 0.2683662165280098], [-0.9716157988656038, 0.45727707819602825, -0.5376127356953474, 0.6626300876609809], [0.014020724262976783, -0.2643422426079685, -0.750808791874266, 0.22371067624615593], [-0.024490099505006135, 1.9131394543817049, -0.023186931500279076, -1.3266491793246087], [1.3822634393800657, -1.3759964147833719, -0.6414035372718514, 0.5336384240398766]], [[2.089943400389317, -2.017714388591036, 1.6711454174837166, -0.8429117726014337], [1.407823181786228, -0.16600172219306608, 0.5610540430931822, 0.04759712891396072], [-0.09339558421005838, 0.5373561080684924, -0.6264275124884702, 0.49340285858683025], [-1.0658353634238578, -0.7171913017637742, -0.49420982623205867, -0.08579065848748894], [1.2279667198701043, -0.8239346781917004, -0.8485469505140043, -1.1130437267387114], [-0.3960882256008402, 0.5714571320641562, -0.7374594503673815, -0.2848802346472962], [-0.4163683144245048, 0.3827639200156128, 0.06306005628402193, -0.04661687555949263]]], "dtype": "float64"}, {"__nd__": [[[-0.5806098743632441, 0.09148518728344766, -2.0073355821449748, 0.6005646180927717], [0.9309711472595998, -0.8293756561812254, -0.06355302824577985, -0.9029276378925517], [1.0076122042921374, 0.638022103029592, 2.905365754466288, -0.19979401144998388], [-0.8672835241573296, -0.4707098575811255, -1.2472258281739477, 0.21930308734331058], [0.32106353464181553, 0.9361309534630777, 0.41775047298731166, -0.3991506082722122], [1.308841835405336, 0.5965915157528774, 0.8488511521670594, -1.1523891832160071], [-0.9736901314693568, -0.9488666230727867, 2.012421855423674, -0.04746628542208381]], [[-0.8151179060392977, -0.6208843954177016, -0.37051363122588304, -2.163180399362114], [1.2257990067736229, -0.18498699546738706, 0.8749902772222725, 0.802044669624543], [1.5252804711174976, 0.6059076613181522, 0.41080477286055705, -0.7030293439623367], [0.16296340347886626, 1.7967351326579435, -1.717788434249212, -0.6591092005876211], [0.32364641617881873, 1.6668239959895288, -1.0442172865679762, -0.39469683180605447], [0.2117067351592105, 3.165755800696517, -0.80280893210664, -0.09042974238578648], [0.04315668501597461, -0.10370055644747933, 1.6240383145038542, 0.5304855907333575]], [[0.8122257890093711, -0.6343643651294132, 0.4578946078635953, 0.487082731292979], [-0.28431569175490745, 0.4291280141007714, 1.8021716158207064, 0.20716255069019446], [-0.343070423147098, -0.6623713896275052, 0.31705955904387295, 1.125269154091136], [0.23556134312775395, 0.4000979885722656, 2.7388393432137237, -0.6714810579558491], [-0.9863119291726956, 0.36592425989631094, -0.12245718104423046, 0.6658942999413479], [1.331332445583354, 0.45100369130048223, -0.10280538971522843, 0.6117676927688177], [-0.24793398770219977, -1.0591302642719698, -0.13947669182016642, -0.3325338931572156]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.5766280062890846, 0.0021278608310701603, -0.48097402514264154, 0.7603778431664915], [-0.108093126532051, -1.051913142309373, 0.7726398837103202, -0.4286010945500254], [-0.4484858356809511, 0.7385463576161955, -0.6942186643905462, -0.08935070946846031], [-0.8946296328413652, -1.4394504017367966, 0.3762050934336387, 1.6374032133317131]], [[-0.7905504412515537, 1.2424164987637685, 1.1459267142565144, 1.5842377400820047], [-0.9880662984708216, -0.5713481207529731, 1.724004292521382, 0.030054856813919305], [0.07493271159241456, 0.7916692976289752, 0.6963565304970232, -0.45971420467486895], [-0.12285455181469986, 0.7264451323418675, 0.3592683694565223, -0.6347515129168594]]], "dtype": "float64"}, {"__nd__": [[0.5476016082665285, 0.37736621923082125, -0.08857347131075646, -0.181300750633184], [0.407061018533132, -0.9237100178933577, 0.08235059705975731, -0.1966118779082676], [-0.22893578905264303, 0.39337812732508326, 0.16352793221611814, 1.2623885788148403], [-0.042470684521735774, 0.19071431780094839, -0.5131963567695214, 0.44839981560654635]], "dtype": "float64"}, {"__nd__": [[-0.19070637416865283, 0.13482784808429094, 0.5648688142104544, 0.519889705544222], [0.24485849685866903, 0.5597406685315276, -0.12767716166503876, -0.32223855420785436], [-0.5573611637348858, 0.27151676434060323, 0.833624861575097, -0.08014018126981994], [0.05324664478624141, -0.0767362583750054, 0.24260827975459598, -0.6795270138105314]], "dtype": "float64"}, {"__nd__": [[0.3896270787362403, 0.479333924700361, -0.2237017471781942, -0.3547711158564169], [-0.30395025415205035, 0.2678317818468259, 0.6245076998051994, -0.3500161321173523], [-0.5697554581258851, 0.08701421929537637, 0.1523781440709453, 0.3281325729084008], [-0.23168667649270536, 0.3369120855334363, 0.48293320944267254, -0.610468453754444]], "dtype": "float64"}, {"__nd__": [[-0.07635759807682996, 0.2865524607526018, 0.34712841616960044, -0.18030671556612132], [-0.7366734808619533, -0.2730403298349267, -0.5311761969284231, -0.17269441135478467], [0.44613889832220305, -0.21810003305652903, 0.0675286383469629, -0.4500234972778565], [0.40397857289068817, -0.21831731148392117, -0.3855276017446385, 0.776945811526655]], "dtype": "float64"}, 1], "e": {"__tup__": [{"__nd__": [[[-0.6465514172292054, 0.10245372005457432, 0.1809568323809183, -0.8604904710528065], [-0.0075220026424999975, 0.06923086204072397, -0.0665038505031981, 0.4019706383831266], [-0.08551780945711898, 0.05309250785591219, 0.07833620057448365, -0.14056818147040756], [0.07797050536147683, 0.039248438069198824, -0.0031288237238106657, 0.21802338623713458]], [[0.24072102597584533, -0.8990779592293783, -0.4888017312336967, -1.224755613135787], [0.861144781144734, -0.5346390096717151, -0.5718423273682027, 0.8200587304427002], [0.4930603732305408, -0.5181200254774749, -0.4603917009931964, 0.16723160369933068], [0.44998910297247324, -0.4757856438139288, -0.40813549302121016, 0.10840583509225582]]], "dtype": "float64"}, {"__nd__": [[[0.19911819977929926, -0.11000449819724167, 0.10896955428510514, -0.17905465397318793], [-0.6904168575875418, -0.36069860359964084, 0.6133564899754486, 0.5120969879645774], [0.6485413557577863, 0.1712904854600519, -0.9480263248141858, -0.3548002491545379], [-0.3043459695759097, -0.9498421247593015, 0.38929805757923597, -1.1440711804209212]], [[-0.09936045820158118, 0.7784126638926183, 0.7344366888996067, -1.9797206476920617], [-0.9107616192897698, 0.012764138094887023, 0.9592846648197733, -0.4881602082032291], [-0.23304351667954615, 0.6775817381424447, 0.41021870701173235, 0.04043245761324482], [-0.032735215547012614, 0.536312510483848, -0.016648085470601703, 0.10457952190646361]]], "dtype": "float64"}, {"__nd__": [[[0.32189128244737525, 0.4912961797400961, 0.1662579558862932, -0.8273256774021668], [-0.06330136994360697, -0.41071870617579254, -0.7219995062820289, 0.9217106489516524], [0.017012662976924497, -0.1076785178257762, 0.4126207754720745, -0.27264328616828093], [-0.504760028881561, -0.22996077507758036, 0.1492641754651664, -0.0549182474535137]], [[-1.7055974464142611, 0.5872817383836382, 1.8924413870811885, -0.7455126376463171], [-1.2005401443429475, -0.46650014988106936, 0.14143591954158868, 1.0978728648102503], [-0.5016755485338402, 0.15366163775950617, 0.3617392530034451, 0.2054572925324339], [-0.326302288739259, -0.04691725960610924, 0.22935541855289426, 0.2947011600605862]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.41246453631946184, 0.7589519583252967, -1.916076233765082, 1.4979461716480538, 1.7902589188972187, -0.06238186965331408, -0.22632982634519216, -0.5087815165390321, 1.1465881578236767, -1.4962526184200344, 0.39555401041219795, 2.06342647402826], [0.20564202087742098, 2.038755808695014, 0.09244606943396169, -0.23539557042329445, 0.9795228520308852, -0.005519108586433316, 0.1670966919961442, -0.33629786662270444, -1.138498373972039, -0.2406761316691224, 0.7727422787142079, 1.145387928168948]], [[-0.004345294154155381, 0.3748207094597816, -0.8601625495940889, -0.10874393490986772, -0.08709393883662168, -0.0683455697114327, 0.12949310049130305, 0.4215789596186413, 1.2878880064239226, 0.8393591882285429, 0.5846618275521462, -1.1451728233792984], [-0.7332705619923161, 0.11895896803106584, 0.17399173542842894, 0.8016340915097359, -1.6809616080231247, 0.25199255014906147, 0.19867111060128373, -0.26123462376356815, -0.7141209917803063, 0.3005465944406616, -0.08273265613519568, 0.5457679826666958]]], "dtype": "float64"}, {"__nd__": [[-0.7697005834617127, -0.32955973399552424, 0.638829032718964, 0.12173398234388362, -0.2763972875923976, -0.09543037697923402, -0.4838870041272616, 0.5994682955184379, 0.6027922292938355, 0.3851639181756397, 0.513155960950791, 0.22091971071037686], [0.5589886444328723, 0.2877943034788532, 0.5979825738369252, 0.027700726858832875, 0.0030671049312372967, -0.1850665628782253, -0.1481344962116347, -0.5877300750911786, 0.5668235219323247, 0.5964655289548866, 0.33021881839574346, 0.4366027997418203], [0.13038097552117353, 0.23476697101815872, -0.49476832008403165, 0.011068822802760121, -0.6779217917349012, -0.2291821670051192, -0.07362363731261812, -0.08421591314125734, -0.38143358246595005, 0.5491000184331669, 0.43236739613460096, -0.1692482118513359], [-0.44762199425590365, -0.15469671526292766, 0.4103728351131502, -0.7178227440838263, 0.3567348886726788, -0.013804333089635818, -0.11972048001216982, 0.44150747307967386, 0.0454649779438947, -0.10219409194885652, 0.5285279655783909, -0.592250430497005], [0.677091371853042, -0.297220341038483, -0.292998425994768, 0.6669473876686145, 1.3554310514812435, 0.526500143652671, -0.2934475474396824, -0.8250218969348663, 0.13484916976770767, 0.04007649694841628, -0.19237220966254412, -0.011844153076980094], [-0.22919079560899336, 0.37248848354384895, 0.004689308194024891, -0.04410099696972264, -0.8324399002846978, 0.13809966620847786, -0.5384572329307302, 0.32878551012709867, 0.023194944877820985, -0.767092059747869, -0.09683467386064822, -0.4367267280008429], [0.255654736389711, 0.21265455665008315, 0.4529701960317021, -0.2713487178691549, 0.2293061451526228, -0.511196048891307, 0.16192227366680223, -0.058954903859807964, 0.1257560612308338, 0.027373178882808942, 0.43582475193007186, -0.1053428267067548], [-0.041320644565242544, 0.585233381587506, 0.003402627984652703, -0.17997967603288845, -0.3266204600808647, 0.035957993733517574, 0.016588830695970787, 0.28479474175838665, -0.03706452953688963, 0.3024188294052067, -0.07471567261370551, 0.6615629341762829], [0.010676904684566009, 0.2989575138359947, 0.18266479552261186, -0.4914313399311615, 0.43887369656830055, -0.13669528376081064, 0.3943206137928536, -0.7625831548021851, -0.7050345293760737, -0.7638702401641828, -0.6846756732146451, -0.1294011180141759], [0.0106933727956911, -0.3649980158773575, -0.11274575402738568, -0.24792834997850602, 0.31449363413240256, 0.3032938965053871, 0.1402365619729669, 0.2775513897009936, 0.7003068952175158, 0.27761547359216104, -0.5812815188481751, -0.06277666122044416], [-0.09778930076081271, 0.3437580233998138, 0.2091941285246251, 0.36146431875131263, -0.5915626257499298, 0.572289989902432, -0.039993418418947144, 0.3995641802033988, 0.20916694835466473, -0.33070904320164424, -0.3065830939387928, 0.2545910477729364], [0.46756863026484397, -0.34486743712125545, -0.43761330438436463, -0.8001281938135861, -0.35782813736651586, -0.284600665689432, -0.10762625967955611, 0.24706159471409372, -0.18743822583434902, -0.43573392288742707, -0.14612581059159788, -0.027127227038534325]], "dtype": "float64"}, {"__nd__": [[0.06111708691014922, 0.05473966058281789], [-0.06822125897413008, 0.2178326636587559], [-0.11519037938701593, -0.4129983815851446], [-0.06921337278487717, 0.7327559141106832], [-0.3244568339715423, 0.9675015158031262], [-0.19992337832929397, 0.28199146697382865], [-0.6364161795973834, -0.04062440347975806], [0.567455889508079, -0.46327512294677886], [-0.2866317927573093, 0.12473720486267353], [0.1385316815692274, -0.5849183849588692], [-0.49638855346386035, 0.11317333938284924], [0.11638163317451672, 0.2895347011429974]], "dtype": "float64"}, {"__nd__": [[-0.06532308310805741, 0.2471219986917037], [0.4761398233623093, -0.48079286216489603], [0.40111011472860425, 0.29834930137504334], [0.5055734656527663, -0.15095529097438531], [-0.011858199496653038, 0.03280855381944881], [-0.1871557475633078, 0.3988131102610348], [0.09120245997144988, -0.3894552125380683], [0.10444007202865452, -0.4162361865651617], [-0.6122176039617576, -0.40620723281952453], [0.009899833543312505, 0.04755502465245395], [-0.04045045969823649, 0.2143275860502355], [0.7558791869102319, -0.4403300078519305]], "dtype": "float64"}, {"__nd__": [[0.1694257475095744, 0.10997462538500438, -0.46454197180029183, -0.19506080866962217, 0.08148249406159244, 0.014028169585004593, -0.32406691153152134, 0.30197096074501967, 0.0954155154935677, 0.25192213890967563, 0.6228828308165945, -0.49436292634399753], [0.38736715674411776, 0.18741246300143327, -0.339062177537281, -0.0209047626951359, -0.16102514072074636, -0.45528604732903566, 0.3158276719931066, -0.3001928406911676, -0.7920165360204703, 0.32134657772647174, -0.18912156567988894, 0.14504469425872374], [0.08069399787540324, -0.1190890921836244, -0.14805761691309502, -0.9858715523094652, -0.1593946781378501, -0.4430174411689928, 0.19677429171965116, -0.3040523421174109, -0.2548402280957109, 0.037933877780391374, -0.4484455142736747, -0.2772155380866934], [-0.1901917218996775, 0.1558656092428435, -0.5656524126282124, 0.2970694680545342, -0.056722363742292295, -0.196498242696162, 0.13229417937748703, 0.39273602025894666, 0.44693532717595047, -0.40700133074871725, 0.5215012824248075, 0.012329656281163193], [0.5077897517831784, -0.4133767912465263, -0.06797711155876002, -0.04471530622206183, -0.49559013630033777, -0.17860371725471647, 0.34950788418487, 0.4314664569102081, 0.25344229874219176, 0.10688980576351965, -0.17478295644888076, 0.7984700756083544], [-0.5261259387248364, -0.471800418875044, -0.3031890393046571, 0.06500942175238669, -0.18277483917563775, 0.2961142324829013, -0.6148678855761297, 0.31857277107061366, 0.6798804779402063, 0.3350354109917942, -0.12092411991579077, -0.23969559982378735], [0.4181517138762824, -0.6677608293473987, -0.3149770820662431, -0.6591519352680248, -0.06233278396504839, 0.7090281635710209, -0.1741656142173915, 0.41085357958854146, -0.4267155119561369, -0.22074942985349538, -0.2294700028657359, 0.4029210379552985], [-0.34782413559699427, -0.2684032780986257, -0.004299420958821147, 0.5947318085034531, 0.016834332470432794, 0.6844467344699585, 0.14750241446662327, -0.10507516544177763, 0.4400749123851663, -0.23224845190907875, -0.36113802231455927, 0.03489316455486613], [-0.6063726230183526, -0.4690760590639014, -0.31990143703743346, 0.006959932667257478, 0.3972784860560723, 0.43890195733311643, -0.08187357750949395, 0.08447990012521728, -0.1265346215154681, -0.5569822620546377, 0.1516505144181206, -0.29057151201189807], [0.2855367719902506, -0.21737032893371644, -0.39854233135750183, 0.12228345562740735, -0.2197423925719909, -0.5165829931548392, 0.395239691059035, 0.1166752733275031, 0.11540142802269011, 0.023427669666814033, -0.22454038461570974, -1.096025073544604], [-0.22542819924003257, -0.1790816182415791, 0.15657130980179432, -0.5157270012561272, 0.17740954717577265, 0.005231820472401792, 0.5644757664592895, -0.28828914455912663, -0.0013041870723417507, 0.09016460455550998, 0.22297355207171418, 0.49637749759821337], [0.18023642688050245, -0.5496850627956227, 0.6386807782458512, -0.7765258938627769, 0.24587519538687916, 0.5092581115138883, -0.2684206894508438, -0.020217298457668603, -0.030506861035068507, -0.4357682617135373, 0.14294140567596833, -0.37462737151430725]], "dtype": "float64"}, 6, {"__nd__": [[[0.31270797079234575, 0.5861544862004814]], [[1.062813187905903, 1.3214986964580127]]], "dtype": "float64"}, {"__nd__": [[[0.4134621114312446, 0.4772355227393223]], [[-0.21258653635183386, -0.06062713295206653]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[0.06991539635744812, -0.9223855571780718, -1.1939697037654367, -0.5446474272683247, -0.28889427648262733, 0.026043673362540654, 0.6436935638865996, 0.38398175232041337, 0.0912100424022268, -0.13135825285984923, -0.07382915707940851, -0.09491270415715886], [0.8781081393586991, -0.4417669465684593, -0.36994607218882625, -2.4779009035150064, -0.009578316409758703, 0.49321530177367323, 0.9613696336061388, -0.4914262543529002, -2.7933150157802804, 0.8304126133752993, -0.8028195285298932, 4.2873670727246616]], [[0.08547283693490199, 1.9556242271911224, 0.9709432518618658, 2.0560288397378956, -0.37617419852245393, -1.1756289315451425, -0.5356677729415648, -0.4151914877556957, 0.441032487227385, 0.6578365749234667, -0.2932469281572616, -0.13078389873047597], [-0.5136149059877214, 0.83303512988434, 0.7466804478864129, -0.19318539285651748, 0.5218664190764153, -0.22186079653075383, 0.17204392523077688, -1.0310854044737656, -0.4959410171201516, -0.016491349173989422, -0.1490835904369233, -0.11255432857002323]]], "dtype": "float64"}, {"__nd__": [[[0.31270797079234575, 0.5861544862004814], [-1.1147241875566376, 5.696662994559787], [-0.6920510946007719, 1.7576867089501977]], [[1.062813187905903, 1.3214986964580127], [-0.396857598071299, -0.5428389069764918], [0.4427966978634331, -1.0572196105317655]]], "dtype": "float64"}, {"__nd__": [[[0.4134621114312446, 0.4772355227393223], [1.066781615782172, -2.087694014356391], [2.374048378866771, -0.649154620523891]], [[-0.21258653635183386, -0.06062713295206653], [-1.8210034983895875, -0.5311916396681998], [1.3992766584933423, -0.18446898501055586]]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(mqa, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")